<a href="https://colab.research.google.com/github/ElenaCoutoFraga/PIIA_AgenteConversacional/blob/main/pruebawebsocket.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**VAD**

In [ ]:
!pip install -q torchaudio
!pip install faster-whisper
!pip install groq
! pip install -U pip
! pip install coqui-tts
! pip install python-dotenv
! pip install fastapi uvicorn websockets

import torch
torch.set_num_threads(1)
from IPython.display import Audio
from pprint import pprint
from faster_whisper import WhisperModel
import matplotlib.pyplot as plt
from TTS.api import TTS
import os
from groq import Groq
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()

In [ ]:
# --------------- DEFINICIÓN DEL VAD ---------------
USE_PIP = True # download model using pip package or torch.hub
USE_ONNX = False # change this to True if you want to test onnx model
if USE_ONNX:
    !pip install -q onnxruntime
if USE_PIP:
  !pip install -q silero-vad
  from silero_vad import (load_silero_vad,
                          read_audio,
                          get_speech_timestamps,
                          save_audio,
                          VADIterator,
                          collect_chunks)
  VAD = load_silero_vad(onnx=USE_ONNX)
else:
  VAD, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                                model='silero_vad',
                                force_reload=True,
                                onnx=USE_ONNX)

  (get_speech_timestamps,
  save_audio,
  read_audio,
  VADIterator,
  collect_chunks) = utils


In [ ]:
# --------------- DEFINICIÓN DEL ASR ---------------
# Especificamos el modelo
model_size = "base"

# Run on GPU with FP16
ASR = WhisperModel(model_size, device="cuda", compute_type="float16")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# --------------- DEFINICIÓN DEL LLM ---------------
load_dotenv("/content/.env", override=True)

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError("No se encontró GROQ_API_KEY en /content/.env")

client = Groq(api_key=api_key.strip())

In [ ]:
# --------------- DEFINICIÓN DEL TTS ---------------
ts = TTS("tts_models/es/css10/vits", gpu=True)

/usr/local/lib/python3.12/dist-packages/TTS/api.py:93: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")


In [ ]:
def run_vad(ruta_audio, SAMPLING_RATE = 16000):
  wav = read_audio(ruta_audio, sampling_rate=SAMPLING_RATE)
  speech_probs = []

  window_size_samples = 512 if SAMPLING_RATE == 16000 else 256
  vad_iterator = VADIterator(VAD, sampling_rate=SAMPLING_RATE)

  speech_events = []
  for i in range(0, len(wav), window_size_samples):
      chunk = wav[i:i + window_size_samples]
      if len(chunk) < window_size_samples:
          break

      speech_prob = VAD(chunk, SAMPLING_RATE).item()
      speech_probs.append(speech_prob)

      speech_dict = vad_iterator(chunk, return_seconds=True)
      if speech_dict:
          speech_events.append(speech_dict)

  vad_iterator.reset_states()

  return {
      "speech_probs": speech_probs,
      "speech_events": speech_events,
  }

In [ ]:
def run_asr(ruta_audio: str):
    segments, info = ASR.transcribe(ruta_audio, beam_size=5)
    text = " ".join(segment.text for segment in segments)
    return {
        "text": text,
        "language": getattr(info, "language", None)
    }

In [ ]:
def run_llm(text: str):
    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "user", "content": text}
        ],
        model="llama-3.3-70b-versatile",
    )

    respuesta = chat_completion.choices[0].message.content
    return respuesta

In [ ]:
#if idioma == "ga": idioma = "es"
def run_tts(text: str, output_path: str = "output.wav"):
    ts.tts_to_file(
        text=text,
        file_path=output_path
    )
    return output_path

In [ ]:
def run_pipeline(ruta_audio: str):
    vad_result = run_vad(ruta_audio)
    asr_result = run_asr(ruta_audio)
    llm_text = run_llm(asr_result["text"])
    tts_path = run_tts(llm_text)

    return {
        "vad": vad_result,
        "asr": asr_result,
        "llm": llm_text,
        "tts_path": tts_path,
    }

In [ ]:
run_pipeline("miau.29.30.mp4")

{'vad': {'speech_probs': [0.0016982493689283729,
   0.05834949389100075,
   0.026905974373221397,
   0.03698375076055527,
   0.01954682171344757,
   0.014430217444896698,
   0.014388549141585827,
   0.012249719351530075,
   0.010167570784687996,
   0.00976551603525877,
   0.005810944363474846,
   0.0067288316786289215,
   0.004778947215527296,
   0.004305916838347912,
   0.006354071199893951,
   0.0057234750129282475,
   0.005845455918461084,
   0.004609226249158382,
   0.003617056179791689,
   0.0026981167029589415,
   0.0021060199942439795,
   0.002492599654942751,
   0.00409600418061018,
   0.003989863209426403,
   0.0044539389200508595,
   0.15187636017799377,
   0.2657926082611084,
   0.11013592034578323,
   0.03634628280997276,
   0.023709462955594063,
   0.013860714621841908,
   0.009211733005940914,
   0.007196311838924885,
   0.006303531117737293,
   0.009656522423028946,
   0.25187599658966064,
   0.08499550074338913,
   0.05802644416689873,
   0.022438600659370422,
   0.0106